In [0]:
-- DDL
-- Customer Transactions Dataset
-- Will be Useful for the Customer Churn Predictor

CREATE SCHEMA IF NOT EXISTS `enterprise-ai-customer-churn-prediction-platform`;

CREATE OR REPLACE VIEW customer_data AS
SELECT * 
FROM `samples`.`bakehouse`.`sales_transactions`;

In [0]:
-- DDL 1
-- Reconfiguring the Dataset to our Needs

CREATE OR REPLACE VIEW data_by_date AS
SELECT LEFT(dateTime, 10) AS Date,
  COUNT(DISTINCT(customerID)) AS total_customers,
  COUNT(DISTINCT(transactionID)) AS unique_transactions,
  COUNT(DISTINCT(product)) AS unique_products_bought,
  SUM(quantity) AS total_products_bought,
  SUM(totalPrice) AS total_spent
FROM customer_data
GROUP BY LEFT(dateTime, 10)
ORDER BY Date;

In [0]:
-- DDL 2
-- Dataset Engineering 2: Customer-Based Data

CREATE OR REPLACE VIEW data_by_customer AS
SELECT customerID,
  COUNT((transactionID)) AS total_purchases,
  COUNT(DISTINCT(product)) AS unique_products_bought,
  SUM(quantity) AS total_products_bought,
  SUM(totalPrice) AS total_spent
FROM customer_data
GROUP BY customerID
ORDER BY customerID;

In [0]:
SELECT *
FROM data_by_customer;

customerID,total_purchases,unique_products_bought,total_products_bought,total_spent
2000000,9,4,67,201
2000001,8,4,51,153
2000002,17,5,109,327
2000003,13,5,72,216
2000004,17,6,121,363
2000005,21,6,110,330
2000006,18,5,160,480
2000007,13,6,122,366
2000008,10,4,58,174
2000009,9,5,52,156


In [0]:
-- DDL 3
-- Data for each Unique Product

CREATE OR REPLACE VIEW data_by_product AS
SELECT product,
  COUNT(DISTINCT(transactionID)) AS total_times_bought,
  COUNT(DISTINCT(paymentMethod)) AS total_payment_methods,
  SUM(quantity) AS total_quantity_bought,
  AVG(unitPrice) AS avg_spent_per_unit,
  SUM(totalPrice) AS total_spent
FROM customer_data
GROUP BY product;

In [0]:
-- DDL 4
-- Rolling Window Date-by-Date KPI
-- Window Functions + Common Table Expressions (CTEs)
--- [Subqueries and especially JOINs have been quite in-demand; let's see if we'll have to use them here again like in my other projects]

CREATE OR REPLACE VIEW kpi_earnings_per_day AS
WITH daily_data AS (
  SELECT
    LEFT(dateTime, 10) AS date,
    franchiseID,
    SUM(totalPrice)  AS total_earnings,
    SUM(unitPrice)   AS total_earnings_per_unit,
    AVG(quantity)    AS avg_quantity_bought
  FROM customer_data
  GROUP BY franchiseID, LEFT(dateTime, 10)
),
with_lag AS (
  SELECT
    franchiseID,
    date,
    total_earnings,
    LAG(total_earnings) OVER (PARTITION BY franchiseID ORDER BY date) AS prev_total_earnings,
    total_earnings_per_unit,
    LAG(total_earnings_per_unit) OVER (PARTITION BY franchiseID ORDER BY date) AS prev_total_earnings_per_unit,
    avg_quantity_bought,
    LAG(avg_quantity_bought) OVER (PARTITION BY franchiseID ORDER BY date) AS prev_avg_quantity_bought
  FROM daily_data
)
SELECT
  franchiseID,
  date,
  total_earnings,
    ROUND((total_earnings - prev_total_earnings)
    / NULLIF(prev_total_earnings, 0) * 100, 2) AS pct_change_earnings_per_day,

  total_earnings_per_unit,
  ROUND((total_earnings_per_unit - prev_total_earnings_per_unit)
    / NULLIF(prev_total_earnings_per_unit, 0) * 100, 2) AS pct_change_total_earnings_per_unit_per_day,

  avg_quantity_bought,
  ROUND((avg_quantity_bought - prev_avg_quantity_bought)
    / NULLIF(prev_avg_quantity_bought, 0) * 100, 2) AS pct_change_avg_quantity_bought_per_day
FROM with_lag
ORDER BY franchiseID, date;

In [0]:
-- DDL 5!
-- GOLD LAYER / ML FEATURE STORE
-- Enterprise Customer Churn Prediction Dataset
-- CTEs, Aggregations, Nested Subqueries, Feature Engineering for Future ML and MLOps

CREATE OR REPLACE TABLE gold_ml_customer_features AS

WITH customer_metrics AS (

SELECT
    customerID,
    COUNT(transactionID)                                   AS total_transactions,
    SUM(TotalPrice)                                        AS lifetime_spend,
    ROUND(AVG(TotalPrice), 2)                              AS avg_transaction_value,
    ROUND(AVG(unitPrice), 2)                               AS avg_unit_price,
    SUM(quantity)                                          AS total_items,
    ROUND(AVG(quantity), 2)                                AS avg_items_per_transaction,
    COUNT(DISTINCT product)                                AS unique_products,
    COUNT(DISTINCT franchiseID)                            AS unique_franchises,
    COUNT(DISTINCT paymentMethod)                          AS payment_method_diversity,
    MIN(dateTime)                                          AS first_purchase,
    MAX(dateTime)                                          AS last_purchase,
    NTILE(5) OVER (ORDER BY SUM(TotalPrice) DESC)          AS customer_value_quintile,
    DATEDIFF(CURRENT_DATE(), MAX(dateTime))
        AS recency_days,
    DATEDIFF(MAX(dateTime), MIN(dateTime))
        AS customer_lifetime_days,
    ROUND( DATEDIFF(MAX(dateTime), MIN(dateTime))
        / NULLIF(COUNT(transactionID)-1, 0), 2)
        AS avg_days_between_purchases,
    COUNT(DISTINCT DATE(dateTime))
        AS active_days,
    COUNT(DISTINCT DATE_TRUNC('month', dateTime))
        AS active_months
FROM customer_data
GROUP BY customerID
)

SELECT
  customerID,
  customer_value_quintile,
  total_transactions,
  lifetime_spend,
  avg_transaction_value,
  avg_unit_price,
  total_items,
  avg_items_per_transaction,
  unique_products,
  unique_franchises,
  payment_method_diversity,
  first_purchase,
  last_purchase,
  recency_days,
  customer_lifetime_days,
  avg_days_between_purchases,
  active_days,
  active_months,
  ROUND(
    lifetime_spend /
    NULLIF(total_transactions,0),2
  ) AS revenue_per_order,
  ROUND(
    total_items /
    NULLIF(total_transactions,0),2
  ) AS avg_basket_size,
  ROUND(
    lifetime_spend /
    NULLIF(active_months,0),2
  ) AS avg_monthly_spend,
CASE
  WHEN unique_franchises > 1
  THEN 1
  ELSE 0
  END AS multi_franchise_customer,
CASE
  WHEN lifetime_spend >
  (
    SELECT AVG(customer_spend)
    FROM
    (
      SELECT
      SUM(TotalPrice) AS customer_spend
      FROM customer_data
      GROUP BY customerID
    )
  )
  THEN 1
  ELSE 0
END AS high_value_customer,

CASE
  WHEN recency_days >= 90
  THEN 1
  ELSE 0
END AS churn_label
FROM customer_metrics
ORDER BY customerID;

num_affected_rows,num_inserted_rows


In [0]:
%python
# Using PySpark to store all Created SQL Tables for Future Analyses

# Read views from the default schema
bydate = spark.read.table("default.data_by_date")
bycustomer = spark.read.table("default.data_by_customer")
byproduct = spark.read.table("default.data_by_product")
kpis = spark.read.table("default.kpi_earnings_per_day")
goldml = spark.read.table("default.gold_ml_customer_features")

# Save each DataFrame as a Delta table (Unity Catalog workspace requires this)
bydate.write.mode("overwrite").saveAsTable("default.data_by_date_export")
bycustomer.write.mode("overwrite").saveAsTable("default.data_by_customer_export")
byproduct.write.mode("overwrite").saveAsTable("default.data_by_product_export")
kpis.write.mode("overwrite").saveAsTable("default.kpi_earnings_per_day_export")
goldml.write.mode("overwrite").saveAsTable("default.gold_ml_customer_features_export")

print("All tables exported successfully!")

All tables exported successfully!
